In [ ]:
# AI Agent using OpenAI API
#
# Objective:
# Build a simple AI Agent using Python and the OpenAI API.
#
# The Agent will:
# 1. Receive a user goal.
# 2. Understand the goal using an LLM.
# 3. Decide whether a tool is required.
# 4. Call the appropriate tool.
# 5. Receive the tool result.
# 6. Provide the final answer.
#
# Example Goal:
# Create a short explanation of what an AI Agent is
# and save it to a file called ai_agent.txt.
#
# Flow:
# User Goal → OpenAI LLM → Tool Selection → Python Tool
# → Tool Result → Final Answer

In [ ]:
# Install the OpenAI Python library.
# This library allows Python to communicate with OpenAI.

!pip install -q openai

In [ ]:
# Load the OpenAI API key stored in Google Colab Secrets.
#
# Secret name:
# OPENAI_API_KEY
#
# IMPORTANT:
# Do not write the actual API key directly in the notebook.

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("OpenAI API key loaded successfully.")

OpenAI API key loaded successfully.


In [ ]:
# Import the OpenAI library.
# Create a client that will communicate with the OpenAI API.

from openai import OpenAI

client = OpenAI()

print("OpenAI client created successfully.")

OpenAI client created successfully.


In [ ]:
# Test the connection to OpenAI.
#
# We send a simple question to the model.
# If this cell works, our API key and OpenAI connection are working.

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What is an AI Agent? Explain in one simple sentence."
        }
    ]
)

print("OpenAI API is working.")
print("\nAI Response:")
print(response.choices[0].message.content)

OpenAI API is working.

AI Response:
An AI agent is a software program that can autonomously perform tasks, make decisions, and learn from its environment using artificial intelligence techniques.


In [ ]:
# Create a Python tool called write_file().
#
# This tool allows the AI Agent to create a text file.
#
# The Agent can provide:
#   filename → name of the file
#   content  → text to be written into the file

def write_file(filename, content):
    """Create a text file and write content into it."""

    with open(filename, "w", encoding="utf-8") as file:
        file.write(content)

    return f"File '{filename}' was created successfully."


print("write_file tool created successfully.")

write_file tool created successfully.


In [ ]:
# Test the Python tool directly before giving it to the AI Agent.
#
# This confirms that our write_file() function works correctly.

result = write_file(
    "test.txt",
    "This file was created by our Python tool."
)

print(result)

File 'test.txt' was created successfully.


In [ ]:
# Tell the OpenAI model about our Python tool.
#
# The schema describes:
#   - Tool name
#   - What the tool does
#   - What parameters the tool requires

TOOL_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": (
                "Create a text file and save the supplied content. "
                "Use this tool whenever the user asks to save "
                "information into a file."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "Name of the file to create."
                    },
                    "content": {
                        "type": "string",
                        "description": "Text to write into the file."
                    }
                },
                "required": ["filename", "content"]
            }
        }
    }
]

print("Tool schema created successfully.")

Tool schema created successfully.


In [ ]:
# Create a dictionary that connects the tool name
# provided by the AI model to the actual Python function.

TOOLS = {
    "write_file": write_file
}

print("Available tools:", list(TOOLS.keys()))

Available tools: ['write_file']


In [ ]:
# Create the AI Agent.
#
# The Agent:
# 1. Receives the user's goal.
# 2. Sends the goal to the OpenAI model.
# 3. Checks whether the model wants to use a tool.
# 4. Executes the requested Python tool.
# 5. Sends the tool result back to the model.
# 6. Produces the final answer.

import json


def run_agent(goal):

    # Store the conversation between the user,
    # AI model, and tool.
    messages = [
        {
            "role": "system",
            "content": """
You are a simple AI Agent.

Understand the user's goal and complete it.

If the user asks you to save or write information
to a file, you MUST use the write_file tool.

After the tool completes, provide a short final answer.
"""
        },
        {
            "role": "user",
            "content": goal
        }
    ]

    # Allow the Agent to perform multiple steps.
    for step in range(5):

        print(f"\n--- Agent Step {step + 1} ---")

        # Send the conversation and available tool
        # to the OpenAI model.
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOL_SCHEMA,
            tool_choice="auto"
        )

        message = response.choices[0].message

        # Add the model response to the conversation.
        messages.append(
            message.model_dump(exclude_none=True)
        )

        # If the model does not request a tool,
        # the Agent has reached its final response.
        if not message.tool_calls:

            print("\nFINAL ANSWER:")
            print(message.content)

            return

        # Process the tool requested by the AI.
        for tool_call in message.tool_calls:

            function_name = tool_call.function.name

            # Convert the AI's tool arguments from JSON to Python.
            arguments = json.loads(
                tool_call.function.arguments
            )

            print("Tool selected:", function_name)
            print("Arguments:", arguments)

            # Find and execute the corresponding Python tool.
            if function_name in TOOLS:

                result = TOOLS[function_name](**arguments)

                print("Tool result:", result)

            else:

                result = f"Unknown tool: {function_name}"

                print(result)

            # Send the tool result back to the AI model.
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                }
            )

    # Stop if the Agent exceeds the maximum number of steps.
    print("\nAgent stopped: maximum steps reached.")

In [ ]:
# Give the AI Agent a goal.
#
# The goal requires the Agent to:
# 1. Create an explanation.
# 2. Save the explanation to ai_agent.txt.
#
# The Agent should therefore select the write_file tool.

goal = """
Create a short and simple explanation of what an AI Agent is.
Save the explanation in a file called ai_agent.txt.
"""

run_agent(goal)


--- Agent Step 1 ---
Tool selected: write_file
Arguments: {'filename': 'ai_agent.txt', 'content': 'An AI Agent is a software program designed to perform tasks autonomously using artificial intelligence. It can learn from data, make decisions, and interact with users or systems to achieve specific goals.'}
Tool result: File 'ai_agent.txt' was created successfully.

--- Agent Step 2 ---

FINAL ANSWER:
The explanation has been saved in the file "ai_agent.txt."


In [ ]:
# Verify that the Agent actually created ai_agent.txt.
#
# This checks the file system and displays the generated content.

import os

filename = "ai_agent.txt"

if os.path.exists(filename):

    print(f"File '{filename}' was created successfully.")

    print("\nFile Content:")
    print("-" * 50)

    with open(filename, "r", encoding="utf-8") as file:
        content = file.read()

    print(content)

else:

    print(f"File '{filename}' was not found.")

File 'ai_agent.txt' was created successfully.

File Content:
--------------------------------------------------
An AI Agent is a software program designed to perform tasks autonomously using artificial intelligence. It can learn from data, make decisions, and interact with users or systems to achieve specific goals.


In [ ]:
# Complete Agent Flow:
#
# User Goal
#     ↓
# OpenAI LLM
#     ↓
# Understand Goal
#     ↓
# Decide to Use Tool
#     ↓
# Tool Calling
#     ↓
# write_file()
#     ↓
# File Created
#     ↓
# Tool Result
#     ↓
# OpenAI LLM
#     ↓
# Final Answer
#
# Main concepts demonstrated:
#
# LLM              → Understands the user's goal.
# Tool             → Performs an actual action.
# Tool Schema      → Describes the tool to the LLM.
# Tool Calling     → LLM requests the tool.
# Agent Loop       → Coordinates LLM and tools.
# Tool Result      → Result returned from Python.
# Final Answer     → LLM provides the completed response.